# Compare Two Allocation Strategies

Compare **VIX-based rebalance** (QQQ/BIL/GLD with high/low vol bands) vs **long-hold QQQ** over the same period. Each engine receives symbols and date range and fetches data itself. Set `ALPACA_API_KEY` and `ALPACA_API_SECRET` (or rely on Yahoo Finance fallback) and optional cache.

In [1]:
from tiportfolio.helpers.cache import enable_data_source_cache
from tiportfolio import (
    FixRatio,
    Schedule,
    ScheduleBasedEngine,
    VixRegimeAllocation,
    VolatilityBasedEngine,
    compare_strategies,
)

from dotenv import load_dotenv
load_dotenv()

enable_data_source_cache("tiportfolio", cache_dir=".cache")

SYMBOLS = ["QQQ", "BIL", "GLD"]
VOLATILITY_SYMBOL = "VIX"
START = "2018-01-01"
END = "2024-12-31"
INITIAL_VALUE = 10_000
FEE_PER_SHARE = 0.0035

## VIX Target Rebalance

In [2]:
# Strategy 1: VIX-based rebalance — high-vol (0.4, 0.4, 0.2), low-vol (0.7, 0.2, 0.1), target 15, bounds -1 / 5
high_weights = {"QQQ": 0.4, "BIL": 0.4, "GLD": 0.2}
low_weights = {"QQQ": 0.7, "BIL": 0.2, "GLD": 0.1}

# API CHANGE NOTE: After the volatility engine refactor:
# - OLD: VixRegimeAllocation(target_vix=X, lower_bound=Y, upper_bound=Z, ...)
# - NEW: VixRegimeAllocation(high_vol_allocation=..., low_vol_allocation=...)
# - VIX parameters (target_vix, lower_bound, upper_bound) are now passed to VolatilityBasedEngine.run()
# This change separates concerns: Allocation strategies decide weights, Engine decides when to use which allocation
allocation_vix = VixRegimeAllocation(
    high_vol_allocation=FixRatio(weights=high_weights),
    low_vol_allocation=FixRatio(weights=low_weights),
)
engine1 = VolatilityBasedEngine(
    allocation=allocation_vix,
    rebalance=Schedule("vix_regime"),
    fee_per_share=FEE_PER_SHARE,
    initial_value=INITIAL_VALUE,
)
result_vix_regime = engine1.run(
    symbols=SYMBOLS,
    start=START,
    end=END,
    volatility_symbol=VOLATILITY_SYMBOL,
    target_vix=15.0,    # These VIX parameters moved here from constructor
    lower_bound=-1.0,    # These VIX parameters moved here from constructor  
    upper_bound=5.0,      # These VIX parameters moved here from constructor
)
print(result_vix_regime.summary())

Loaded cached bar data.

Loaded cached bar data.

Loaded cached bar data.

Backtest Summary
----------------
Sharpe Ratio:    0.6798
CAGR:            11.73%
Max Drawdown:    17.22%
MAR Ratio:       0.6809
Kelly Leverage:  5.9507
Final Value:      21,716.22
Total Fee:        3.94
Rebalances:      113


In [3]:
from tiportfolio.report import plot_portfolio_with_assets_interactive

fig = plot_portfolio_with_assets_interactive(result_vix_regime, mark_rebalances=True)
fig.show()

In [4]:
from tiportfolio.report import rebalance_decisions_table, plot_strategy_comparison_interactive

decisions_df = rebalance_decisions_table(result_vix_regime)
decisions_df.head(10)

,date,equity_before,equity_after,fee_paid,QQQ_price,QQQ_qty_before,QQQ_trade_qty,QQQ_qty_after,QQQ_value_after,BIL_price,BIL_qty_before,BIL_trade_qty,BIL_qty_after,BIL_value_after,GLD_price,GLD_qty_before,GLD_trade_qty,GLD_qty_after,GLD_value_after
0,2018-01-31 05:00:00+00:00,10504.665,10504.657,0.008,160.26,46.688,-0.805,45.883,7353.266,75.17,26.638,1.311,27.949,2100.933,127.65,7.990,0.239,8.229,1050.467
1,2018-02-05 05:00:00+00:00,10006.896,10006.714,0.182,149.58,45.883,-19.123,26.760,4002.758,75.17,27.949,25.300,53.249,4002.758,126.71,8.229,7.566,15.795,2001.379
2,2018-02-20 05:00:00+00:00,10181.697,10181.690,0.007,156.31,26.760,-0.705,26.055,4072.679,75.21,53.249,0.901,54.151,4072.679,126.24,15.795,0.336,16.131,2036.339
3,2018-03-01 05:00:00+00:00,10140.303,10140.302,0.001,155.60,26.055,0.012,26.068,4056.121,75.24,54.151,-0.242,53.909,4056.121,124.72,16.131,0.130,16.261,2028.061
4,2018-03-22 04:00:00+00:00,10128.818,10128.816,0.002,154.27,26.068,0.195,26.263,4051.527,75.29,53.909,-0.097,53.812,4051.527,125.98,16.261,-0.181,16.080,2025.764
5,2018-04-02 04:00:00+00:00,9970.078,9970.071,0.007,147.36,26.263,0.801,27.063,3988.031,75.33,53.812,-0.871,52.941,3988.031,127.26,16.080,-0.411,15.669,1994.016
6,2018-04-06 04:00:00+00:00,9985.819,9985.818,0.001,148.42,27.063,-0.151,26.912,3994.328,75.34,52.941,0.077,53.017,3994.328,126.40,15.669,0.132,15.800,1997.164
7,2018-05-09 04:00:00+00:00,10245.830,10245.649,0.181,159.08,26.912,18.172,45.085,7172.081,75.45,53.017,-25.858,27.159,2049.166,124.33,15.800,-7.560,8.241,1024.583
8,2018-05-16 04:00:00+00:00,10276.450,10276.449,0.001,160.12,45.085,-0.159,44.926,7193.515,75.47,27.159,0.074,27.233,2055.290,122.29,8.241,0.163,8.403,1027.645
9,2018-06-01 04:00:00+00:00,10440.318,10440.315,0.003,163.69,44.926,-0.279,44.647,7308.222,75.53,27.233,0.412,27.645,2088.064,122.51,8.403,0.119,8.522,1044.032


## Long Hold QQQ

In [5]:
# Strategy 2: Long hold QQQ (no rebalance)
engine2 = ScheduleBasedEngine(
    allocation=FixRatio(weights={"QQQ": 1.0}),
    rebalance=Schedule("month_start"),
    fee_per_share=FEE_PER_SHARE,
    initial_value=INITIAL_VALUE,
)
result_qqq_only = engine2.run(
    symbols=["QQQ"],
    start=START,
    end=END,
)
print(result_qqq_only.summary())

Loaded cached bar data.

Backtest Summary
----------------
Sharpe Ratio:    0.6860
CAGR:            19.24%
Max Drawdown:    35.01%
MAR Ratio:       0.5495
Kelly Leverage:  2.8433
Final Value:      34,218.64
Total Fee:        0.00
Rebalances:      0


In [6]:
compare_strategies(
    result_vix_regime,
    result_qqq_only,
    names=['vix', 'long_qqq']
)

,vix,long_qqq,better
metric,,,
sharpe_ratio,0.679788,0.685981,long_qqq
cagr,0.117285,0.192355,long_qqq
max_drawdown,0.172240,0.350069,vix
mar_ratio,0.680936,0.549477,vix
final_value,21716.221884,34218.635363,long_qqq
kelly_leverage,5.950668,2.843300,vix
total_fee,3.936685,0.000000,long_qqq


In [7]:
# Interactive comparison chart
plot_strategy_comparison_interactive(
    result_vix_regime,
    result_qqq_only,
    names=[
        "VIX regime (QQQ/BIL/GLD)",
        "Long QQQ",
    ]
)

In [8]:
## VIX Target Rebalance with Freezing (30 days)

engine_freezing = VolatilityBasedEngine(
    allocation=allocation_vix,
    rebalance=Schedule("vix_regime"),
    fee_per_share=FEE_PER_SHARE,
    initial_value=INITIAL_VALUE,
    freezing_days=30,
)
result_vix_freezing = engine_freezing.run(
    symbols=SYMBOLS,
    start=START,
    end=END,
    volatility_symbol=VOLATILITY_SYMBOL,
    target_vix=15.0,
    lower_bound=-1.0,
    upper_bound=5.0,
)
print(result_vix_freezing.summary())

Loaded cached bar data.

Loaded cached bar data.

Loaded cached bar data.

Backtest Summary
----------------
Sharpe Ratio:    0.7283
CAGR:            13.94%
Max Drawdown:    20.90%
MAR Ratio:       0.6670
Kelly Leverage:  5.2869
Final Value:      24,912.29
Total Fee:        3.06
Rebalances:      45


In [11]:
compare_strategies(
    result_vix_regime,
    result_qqq_only,
    result_vix_freezing,
    names=["VIX Original",
           "qqq_only",
           "VIX with 30-day Freezing"])

,VIX Original,qqq_only,VIX with 30-day Freezing,better
metric,,,,
sharpe_ratio,0.679788,0.685981,0.728269,VIX with 30-day Freezing
cagr,0.117285,0.192355,0.139440,qqq_only
max_drawdown,0.172240,0.350069,0.209049,VIX Original
mar_ratio,0.680936,0.549477,0.667021,VIX Original
final_value,21716.221884,34218.635363,24912.287765,qqq_only
kelly_leverage,5.950668,2.843300,5.286929,VIX Original
total_fee,3.936685,0.000000,3.055220,qqq_only


In [10]:
# Interactive comparison chart
plot_strategy_comparison_interactive(
    result_vix_regime,
    result_qqq_only,
    result_vix_freezing,
    names=[
        "VIX regime (QQQ/BIL/GLD)",
        "Long QQQ",
        "Result freezing",
    ]
)